# IPN 13 lớp: fixed-64 vs variable-length (bước 6)

So sánh có kiểm soát **c_ref fixed-64** (run đã có trên Drive) với **variable-length** (giữ nguyên chuỗi, mask padding, clip > `max_frames` dùng sliding window stride `max_frames/2`, gộp mean logits theo clip). Cùng recipe neural C, cùng 3 seed, **chỉ validation**; test khóa.

Báo cáo: macro-F1 (mean±sd, hiệu theo cặp seed), ensemble 3 seed, F1 từng lớp, nhầm lẫn G01↔G08 và G02↔G09, accuracy theo nhóm độ dài, thời gian và VRAM.

Khác biệt cố ý: batch 32 (window dài hơn), positional capacity = `max_frames`, loss có trọng số `1/số window` để clip dài không lấn át. Mọi thứ khác (d192, 4 lớp, label smoothing, class weight, jitter, keypoint dropout, patience 12, chọn macro-F1) giữ như C.

Điều kiện: `STEP5_WINDOW_REPORT.json` PASS trên Drive.

In [ ]:
from pathlib import Path
import csv
import datetime as dt
import hashlib
import json
import os
import shutil
import statistics
import subprocess
import sys
import uuid
import zipfile

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Không chạy trên Colab: bỏ qua mount Drive.')

# --- Chỉ sửa các biến này ---
DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/ipn-extract-data')
REPO_URL = 'https://github.com/HuyenHoang28/DynamicHandGestureWithXAI.git'
REPO_REF = 'HuyenHoang'      # nhánh chứa code mới (hand_geometry.py, ensemble_neural_only.py)
REPO_COMMIT = None           # đặt hash để ghim chính xác; None thì dùng đầu nhánh và ghi lại hash đã dùng
SEEDS = [42, 123, 3407]
RUN_LOCKED_TEST = False      # test giữ khóa; ablation này chỉ dùng validation
MAX_FRAMES_ARMS = [256]      # thêm 128 hoặc 384 để so nhiều giới hạn; stride = max_frames // 2
NUM_CLASSES = 13
EXPECTED_COUNTS = {'train': 2443, 'val': 589, 'test': 1101}   # None để bỏ kiểm tra khi chạy thử nhỏ
DEVICE = 'cuda:0'

RUN_ID = dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:10]
WORK_ROOT = Path('/content') / f'ipn13_variable_length_{RUN_ID}'
REPO_DIR = WORK_ROOT / 'repo'
FIXED_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_neural_sweep'          # run fixed-64 (c_ref) đã có
OUTPUT_ROOT = DRIVE_DATA_ROOT / 'outputs' / 'ipn13_variable_length'      # run variable-length mới
DRIVE_RUN_ROOT = DRIVE_DATA_ROOT / 'runs' / 'ipn13_variable_length'
TRAIN_FILE = DRIVE_DATA_ROOT / 'train.txt'
VAL_FILE = DRIVE_DATA_ROOT / 'val.txt'
TEST_FILE = DRIVE_DATA_ROOT / 'test.txt'
print({'run_id': RUN_ID, 'seeds': SEEDS, 'locked_test': RUN_LOCKED_TEST})


## 1. Clone code (phải push nhánh trước)

In [ ]:
COMMANDS = []
WORK_ROOT.mkdir(parents=True, exist_ok=True)


def run_argv(argv, cwd=None, log_path=None):
    command = [str(value) for value in argv]
    COMMANDS.append(command)
    print('$', ' '.join(command))
    log = open(log_path, 'a', encoding='utf-8') if log_path else None
    try:
        process = subprocess.Popen(command, cwd=str(cwd) if cwd else None, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            if log:
                log.write(line)
        if process.wait() != 0:
            raise subprocess.CalledProcessError(process.returncode, command)
    finally:
        if log:
            log.close()


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


if not REPO_DIR.exists():
    run_argv(['git', 'clone', '--filter=blob:none', REPO_URL, REPO_DIR])
run_argv(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_COMMIT or REPO_REF])
run_argv(['git', '-C', REPO_DIR, 'checkout', '--detach', REPO_COMMIT or 'FETCH_HEAD'])
REPO_COMMIT_RESOLVED = subprocess.run(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], check=True, text=True,
                                      capture_output=True).stdout.strip()
if REPO_COMMIT:
    assert REPO_COMMIT_RESOLVED == REPO_COMMIT, (REPO_COMMIT_RESOLVED, REPO_COMMIT)
NEURAL_DIR = REPO_DIR / 'final' / 'code' / 'neural_only_experiments'
for required in ('train_neural_only.py', 'test_neural_only.py', 'hand_geometry.py', 'ensemble_neural_only.py', 'train_variable_length.py', 'variable_dataset.py'):
    assert (NEURAL_DIR / required).is_file(), f'{required} chưa có trên nhánh {REPO_REF}: hãy commit và push code mới trước.'
if (REPO_DIR / 'requirements.txt').is_file():
    run_argv([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_DIR / 'requirements.txt'])
print('Repository revision:', REPO_COMMIT_RESOLVED)


## 2. Kiểm tra dữ liệu, cache variable_v1 và báo cáo bước 5

In [ ]:
def read_manifest(path):
    rows = []
    for line in Path(path).read_text(encoding='utf-8').splitlines():
        if line.strip():
            source, _, label = line.rsplit(maxsplit=2)
            rows.append((source, int(label)))
    return rows


manifests = {name: read_manifest(path) for name, path in (('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE))}
for name, rows in manifests.items():
    ids = {label for _, label in rows}
    if EXPECTED_COUNTS:
        assert len(rows) == EXPECTED_COUNTS[name], (name, len(rows))
    assert ids == set(range(NUM_CLASSES)), (name, sorted(ids))
sources = {name: {source.replace(chr(92), '/') for source, _ in rows} for name, rows in manifests.items()}
assert not (sources['train'] & sources['val']) and not (sources['train'] & sources['test']) and not (sources['val'] & sources['test'])

CACHE_ROOT = None
for candidate in (DRIVE_DATA_ROOT / 'keypoint_tensor_cache', DRIVE_DATA_ROOT / 'data' / 'keypoint_tensor_cache'):
    if candidate.is_dir():
        CACHE_ROOT = candidate
        break
assert CACHE_ROOT is not None, 'Không thấy keypoint_tensor_cache trên Drive.'
print({'counts': {name: len(rows) for name, rows in manifests.items()}, 'cache_root': str(CACHE_ROOT)})

VAR_CACHE = CACHE_ROOT / 'variable_v1'
assert VAR_CACHE.is_dir(), 'Chưa có cache variable_v1 (bước 3).'
step5 = VAR_CACHE / 'STEP5_WINDOW_REPORT.json'
assert step5.is_file() and json.loads(step5.read_text())['status'] == 'PASS', 'Bước 5 chưa PASS'
print('Bước 5 PASS; variable cache:', VAR_CACHE)


## 3. Huấn luyện variable-length (chỉ khi thiếu run)

Run hoàn tất được dùng lại; run dở dang được đổi tên, không xóa.

In [ ]:
import time
VAR_BASE = ['--cache-root', CACHE_ROOT, '--num-classes', NUM_CLASSES, '--batch-size', 32, '--eval-batch-size', 64,
            '--epochs', 80, '--d-model', 192, '--layers', 4, '--heads', 8, '--dropout', 0.15,
            '--label-smoothing', 0.05, '--pooling', 'masked_mean', '--class-weight-power', 0.5,
            '--jitter-std', 0.005, '--frame-mask-probability', 0.0, '--keypoint-dropout-probability', 0.02,
            '--select-metric', 'macro_f1', '--early-stop-patience', 12, '--num-workers', 0, '--log-every', 20]
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VAR_RUNS = {}
for max_frames in MAX_FRAMES_ARMS:
    arm = f'var{max_frames}'
    for seed in SEEDS:
        run_dir = OUTPUT_ROOT / f'{arm}_seed{seed}'
        VAR_RUNS[(arm, seed)] = run_dir
        marker = run_dir / 'RUN_COMPLETE.json'
        if marker.is_file() and (run_dir / 'best.pt').is_file():
            print('Dùng lại run đã hoàn tất:', run_dir.name)
            continue
        if run_dir.exists() and any(run_dir.iterdir()):
            stale = run_dir.with_name(run_dir.name + '_incomplete_' + dt.datetime.now().strftime('%Y%m%dT%H%M%S'))
            run_dir.rename(stale)
            print('Run dở dang được đổi tên thành', stale.name)
        command = [sys.executable, '-u', NEURAL_DIR / 'train_variable_length.py', '--train-file', TRAIN_FILE,
                   '--val-file', VAL_FILE, '--output-dir', run_dir, '--seed', seed, '--device', DEVICE,
                   '--max-frames', max_frames, '--stride', max_frames // 2] + VAR_BASE
        assert '--test-file' not in command and '--eval-file' not in command
        run_dir.mkdir(parents=True)
        started = time.time()
        run_argv(command, cwd=NEURAL_DIR, log_path=run_dir / 'train.log')
        assert (run_dir / 'best.pt').is_file()
        marker.write_text(json.dumps({'arm': arm, 'seed': seed, 'commit': REPO_COMMIT_RESOLVED,
                                      'train_seconds': time.time() - started,
                                      'command': [str(v) for v in command]}, indent=2), encoding='utf-8')
print('Xong huấn luyện:', len(VAR_RUNS), 'run')


## 4. Đánh giá validation

Variable-length: metric tính một lần mỗi clip sau khi gộp window. Fixed-64: dùng lại `val_eval/metrics.json` của c_ref.

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


FIXED_RUNS = {('fixed64', s): FIXED_ROOT / f'c_ref_seed{s}' for s in SEEDS}
for run_dir in FIXED_RUNS.values():
    assert (run_dir / 'best.pt').is_file(), f'Thiếu {run_dir}: cần run c_ref trong ipn13_neural_sweep.'
RUN_DIRS = {**FIXED_RUNS, **VAR_RUNS}
ARMS = ['fixed64'] + [f'var{m}' for m in MAX_FRAMES_ARMS]

val_metrics = {}
for (arm, seed), run_dir in RUN_DIRS.items():
    metrics_path = run_dir / 'val_eval' / 'metrics.json'
    if not metrics_path.is_file():
        if arm == 'fixed64':
            run_argv([sys.executable, '-u', NEURAL_DIR / 'test_neural_only.py', '--checkpoint', run_dir / 'best.pt',
                      '--cache-root', CACHE_ROOT, '--test-file', VAL_FILE, '--split', 'val', '--batch-size', 128,
                      '--num-workers', 0, '--device', DEVICE, '--timing-warmup-batches', 1, '--timing-repeats', 1,
                      '--output', metrics_path], cwd=NEURAL_DIR)
        else:
            run_argv([sys.executable, '-u', NEURAL_DIR / 'train_variable_length.py', '--checkpoint', run_dir / 'best.pt',
                      '--cache-root', CACHE_ROOT, '--eval-file', VAL_FILE, '--split', 'val', '--eval-batch-size', 64,
                      '--device', DEVICE, '--output', metrics_path], cwd=NEURAL_DIR)
    val_metrics[(arm, seed)] = read_json(metrics_path)
print('Đánh giá validation xong.')


## 5. Bảng kết quả, ensemble, hiệu theo cặp seed

In [ ]:
import torch


def stats(values):
    return statistics.mean(values), (statistics.stdev(values) if len(values) > 1 else 0.0)


def macro_f1(pred, labels, k=NUM_CLASSES):
    scores = []
    for c in range(k):
        tp = sum(p == c and y == c for p, y in zip(pred, labels))
        pp = sum(p == c for p in pred)
        sup = sum(y == c for y in labels)
        pr = tp / pp if pp else 0.0
        rc = tp / sup if sup else 0.0
        scores.append(2 * pr * rc / (pr + rc) if pr + rc else 0.0)
    return sum(scores) / k


def variable_ensemble(arm):
    '''Trung bình softmax qua seed; clip_logits.pt đã gộp window, sắp theo đường dẫn để khớp thứ tự.'''
    probs, labels, paths = [], None, None
    for seed in SEEDS:
        saved = torch.load(RUN_DIRS[(arm, seed)] / 'val_eval' / 'clip_logits.pt')
        order = sorted(range(len(saved['paths'])), key=lambda i: saved['paths'][i])
        ordered_paths = [saved['paths'][i] for i in order]
        assert paths is None or paths == ordered_paths, 'Các seed đánh giá khác tập clip'
        paths = ordered_paths
        probs.append(saved['logits'][order].softmax(1))
        labels = saved['labels'][order]
    pred = torch.stack(probs).mean(0).argmax(1)
    return {'accuracy': float((pred == labels).float().mean()),
            'macro_f1': macro_f1(pred.tolist(), labels.tolist())}


ENSEMBLE_ROOT = OUTPUT_ROOT / 'ensembles'
fixed_out = ENSEMBLE_ROOT / 'fixed64' / 'val' / 'metrics.json'
if not fixed_out.is_file():
    run_argv([sys.executable, '-u', NEURAL_DIR / 'ensemble_neural_only.py', '--checkpoints',
              *[FIXED_RUNS[('fixed64', s)] / 'best.pt' for s in SEEDS], '--cache-root', CACHE_ROOT,
              '--test-file', VAL_FILE, '--split', 'val', '--device', DEVICE, '--output', fixed_out], cwd=NEURAL_DIR)
ensemble_val = {'fixed64': {k: read_json(fixed_out)[k] for k in ('accuracy', 'macro_f1')}}
for arm in ARMS[1:]:
    ensemble_val[arm] = variable_ensemble(arm)

summary = []
print(f"{'arm':9} {'acc mean±sd':>16} {'macroF1 mean±sd':>18} {'ens acc':>8} {'ens F1':>8}")
for arm in ARMS:
    accs = [val_metrics[(arm, s)]['accuracy'] for s in SEEDS]
    f1s = [val_metrics[(arm, s)]['macro_f1'] for s in SEEDS]
    row = {'arm': arm, 'acc_mean': stats(accs)[0], 'acc_std': stats(accs)[1], 'f1_mean': stats(f1s)[0],
           'f1_std': stats(f1s)[1], 'ensemble_acc': ensemble_val[arm]['accuracy'],
           'ensemble_f1': ensemble_val[arm]['macro_f1']}
    summary.append(row)
    print(f"{arm:9} {row['acc_mean']:.4f}±{row['acc_std']:.4f}   {row['f1_mean']:.4f}±{row['f1_std']:.4f}   "
          f"{row['ensemble_acc']:8.4f} {row['ensemble_f1']:8.4f}")

print('\nHiệu theo cặp seed (variable - fixed64), macro-F1 validation:')
verdicts = {}
for arm in ARMS[1:]:
    deltas = [val_metrics[(arm, s)]['macro_f1'] - val_metrics[('fixed64', s)]['macro_f1'] for s in SEEDS]
    for s, d in zip(SEEDS, deltas):
        print(f'  {arm} seed {s}: {d:+.4f}')
    mean_delta = statistics.mean(deltas)
    print(f'  {arm} trung bình: {mean_delta:+.4f}')
    if all(d > 0 for d in deltas) and mean_delta > 0.01:
        verdicts[arm] = 'có dấu hiệu tốt hơn fixed-64 (cả 3 seed dương, trung bình > 0.01)'
    elif all(d < 0 for d in deltas):
        verdicts[arm] = 'kém hơn fixed-64 ở mọi seed'
    else:
        verdicts[arm] = 'lẫn lộn giữa các seed; chưa chứng minh được khác biệt'
for arm, text in verdicts.items():
    print(f'KẾT LUẬN {arm}: {text}')


## 6. Lớp click/pinch và nhầm lẫn G01↔G08, G02↔G09

In [ ]:
names = [row['label_name'] for row in val_metrics[('fixed64', SEEDS[0])]['per_class']]
idx = {n: i for i, n in enumerate(names)}
print('F1 trung bình qua seed (validation) các lớp khó:')
print(f"{'class':6}" + ''.join(f'{a:>10}' for a in ARMS))
for name in ['G01', 'G02', 'G08', 'G09', 'G10', 'G11']:
    print(f'{name:6}' + ''.join(
        f"{statistics.mean(val_metrics[(a, s)]['per_class'][idx[name]]['f1'] for s in SEEDS):10.3f}" for a in ARMS))

print('\nSố clip nhầm (cộng 3 seed):')
for a, b in (('G01', 'G08'), ('G02', 'G09')):
    for arm in ARMS:
        ab = sum(val_metrics[(arm, s)]['confusion_matrix'][idx[a]][idx[b]] for s in SEEDS)
        ba = sum(val_metrics[(arm, s)]['confusion_matrix'][idx[b]][idx[a]] for s in SEEDS)
        print(f'  {arm:9} {a}->{b}: {ab:3d}   {b}->{a}: {ba:3d}   tổng {ab + ba}')


## 7. Accuracy theo nhóm độ dài, thời gian, VRAM

In [ ]:
import csv as _csv
sys.path[:0] = [str(REPO_DIR / 'final' / 'code')]
from neural_only_experiments.variable_dataset import variable_cache_path
from src_neurosymbolic.variable_keypoint_cache import load_variable_cache

BINS = (('<64', 'lt64', 0, 64), ('64-127', '64_127', 64, 128), ('128-255', '128_255', 128, 256),
        ('>=256', 'ge256', 256, 10**9))
lengths = {src: int(load_variable_cache(variable_cache_path(src, CACHE_ROOT))['length'])
           for src, _ in manifests['val']}
print('Phân bố độ dài validation:', {n: sum(lo <= v < hi for v in lengths.values()) for n, _, lo, hi in BINS})


def fixed_group_acc(seed):
    rows = list(_csv.DictReader((FIXED_RUNS[('fixed64', seed)] / 'val_eval' / 'predictions.csv').open(encoding='utf-8')))
    out = {}
    for name, _, lo, hi in BINS:
        sel = [r for r in rows if lo <= lengths[r['path']] < hi]
        out[name] = sum(r['true_id'] == r['pred_id'] for r in sel) / max(len(sel), 1)
    return out


print(f"\n{'nhóm':9}" + ''.join(f'{a:>10}' for a in ARMS))
group_table = {}
for name, key, lo, hi in BINS:
    values = []
    for arm in ARMS:
        if arm == 'fixed64':
            values.append(statistics.mean(fixed_group_acc(s)[name] for s in SEEDS))
        else:
            values.append(statistics.mean(
                val_metrics[(arm, s)]['length_groups'][key].get('accuracy', float('nan')) for s in SEEDS))
    group_table[name] = dict(zip(ARMS, values))
    print(f'{name:9}' + ''.join(f'{v:10.3f}' for v in values))

print('\nThời gian suy luận / VRAM (phạm vi khác nhau, chỉ để tham khảo):')
fixed_ms = statistics.mean(val_metrics[('fixed64', s)]['inference_timing']['latency_ms_per_sample'] for s in SEEDS)
print(f'  fixed64: {fixed_ms:.3f} ms/clip (forward, batch 128)')
timing = {'fixed64_ms_per_clip': fixed_ms}
for arm in ARMS[1:]:
    ms = statistics.mean(val_metrics[(arm, s)]['timing']['ms_per_clip'] for s in SEEDS)
    peak = max(val_metrics[(arm, s)]['timing']['peak_cuda_bytes'] for s in SEEDS) / 2**20
    train_s = statistics.mean(read_json(RUN_DIRS[(arm, s)] / 'RUN_COMPLETE.json').get('train_seconds', float('nan')) for s in SEEDS)
    print(f'  {arm}: {ms:.3f} ms/clip (gồm mọi window), peak eval VRAM {peak:.0f} MiB, train {train_s / 60:.1f} phút/run')
    timing[arm] = {'ms_per_clip': ms, 'peak_eval_mib': peak, 'train_minutes_per_run': train_s / 60}


## 8. Đóng gói ZIP (từ chối ghi đè)

In [ ]:
DRIVE_RUN_ROOT.mkdir(parents=True, exist_ok=True)
run_dir_drive = DRIVE_RUN_ROOT / RUN_ID
run_zip = DRIVE_RUN_ROOT / f'{RUN_ID}.zip'
if run_dir_drive.exists() or run_zip.exists():
    raise FileExistsError(f'Từ chối ghi đè: {run_dir_drive} hoặc {run_zip}')
payload = run_dir_drive / 'payload'
payload.mkdir(parents=True)
(payload / 'summary_validation.json').write_text(json.dumps(
    {'summary': summary, 'verdicts': verdicts, 'length_groups': group_table, 'timing': timing,
     'commit': REPO_COMMIT_RESOLVED}, indent=2), encoding='utf-8')
(payload / 'commands.json').write_text(json.dumps(COMMANDS, indent=2), encoding='utf-8')
for (arm, seed), run_dir in RUN_DIRS.items():
    target = payload / arm
    target.mkdir(exist_ok=True)
    shutil.copy2(run_dir / 'val_eval' / 'metrics.json', target / f'seed{seed}_val_metrics.json')
    shutil.copy2(run_dir / 'history.json', target / f'seed{seed}_history.json')
with zipfile.ZipFile(run_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(payload.rglob('*')):
        if path.is_file():
            archive.write(path, path.relative_to(payload).as_posix())
print('ZIP:', run_zip, run_zip.stat().st_size, 'bytes')
